# Vision2Slope Tutorial

This tutorial demonstrates how to use Vision2Slope for road slope analysis from street view images.


## 1. Installation

Install Vision2Slope from the local source so you can run the tutorial.


In [1]:
# Install Vision2Slope package from source
import subprocess
import sys

# Navigate to package directory and install in editable mode
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", "../src/Vision2Slope"],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("✓ Vision2Slope package installed successfully!")
else:
    print(f"✗ Installation error: {result.stderr}")


✓ Vision2Slope package installed successfully!


## 2. Environment Setup and Imports

First, import the necessary libraries and modules.

In [2]:
# Import required libraries
import sys
import os
from pathlib import Path
import glob

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Import Vision2Slope modules
from vision2slope import (
    PipelineConfig,
    VisualizationConfig,
    ProcessingConfig,
    Vision2SlopePipeline
)

print("✓ All libraries imported successfully!")
print(f"Working directory: {Path.cwd()}")


/Users/cubics/miniconda3/envs/zensvi/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ All libraries imported successfully!
Working directory: /Users/cubics/Vision2Slope/examples


## 3. (Optional) Download Mapillary Panoramas

Download panoramic street view images from Mapillary for analysis.

**Note:** Requires a Mapillary API key. Register at https://www.mapillary.com/


In [3]:
# Set your Mapillary API key (get it from https://www.mapillary.com/)
mly_api_key = "MLY|9492953287469082|9322fd1bba7d9148f1d62e0a4a14ef74"  # Replace with your actual API key

# Configure download options
from zensvi.download import MLYDownloader

downloader = MLYDownloader(mly_api_key=mly_api_key)

# Example: Download panoramic images from San Francisco
downloader.download_svi(
    "sf",
    lat=37.797423890238,
    lon=-122.44403351517,
    buffer=5,
    resolution=2048,
    image_type="pano"  # 'flat', 'pano', or 'all'
)

print("Configuration ready. Replace 'YOUR_MAPILLARY_API_KEY_HERE' and uncomment to download images.")


/Users/cubics/miniconda3/envs/zensvi/lib/python3.10/site-packages/zensvi/download/base.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Getting pids...


Loading cache files: 0it [00:00, ?it/s]


[Vector Tiles API] Fetching 1 tile for images ...


Filtering data: 100%|██████████| 156396/156396 [00:00<00:00, 444507.34it/s]


...

Getting urls by batch size 15:   0%|          | 0/1 [00:00<?, ?it/s]

Error: HTTPSConnectionPool(host='graph.mapillary.com', port=443): Max retries exceeded with url: /663520818838910/?fields=altitude,atomic_scale,camera_parameters,camera_type,captured_at,compass_angle,computed_altitude,computed_compass_angle,computed_geometry,computed_rotation,exif_orientation,geometry,height,thumb_256_url,thumb_1024_url,thumb_2048_url,merge_cc,mesh,quality_score,sequence,sfm_cluster,width&access_token=MLY%7C9492953287469082%7C9322fd1bba7d9148f1d62e0a4a14ef74 (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1017)')))
Error: HTTPSConnectionPool(host='graph.mapillary.com', port=443): Max retries exceeded with url: /146344068155205/?fields=altitude,atomic_scale,camera_parameters,camera_type,captured_at,compass_angle,computed_altitude,computed_compass_angle,computed_geometry,computed_rotation,exif_orientation,geometry,height,thumb_256_url,thumb_1024_url,thumb_2048_url,merge_cc,mesh,quality_score,sequence,s

Getting urls by batch size 15: 100%|██████████| 1/1 [00:01<00:00,  1.16s/it]

Error: HTTPSConnectionPool(host='scontent-sin6-3.xx.fbcdn.net', port=443): Max retries exceeded with url: /m1/v/t6/An8lajKHxy7hN5LRqnrLqEGPnC5pavvDAzBXjUt6975cSZ5I_MlsHqojY393sJ7qx9H0Gcf7nbF77czg0kaitQAhIywa_4KWpMUvYFesdTl3cTwmr9nqIsFvh8FhxaTjQDVQvXXsrnujb-5cQLe-Pq4?stp=s2048x1024&edm=ALXxkZ8EAAAA&_nc_gid=HUwn-2SmGiwmHcbmIVPw6A&_nc_oc=AdkXovHb9T6hECq_IeNYZ21aQHmy5xeATUODG7cOyKMIB-m531ol_mP0Cz6vXQR7OzQ&ccb=10-5&oh=00_AftPMwynjQpNLV-CYY8m0W1NuCUE_vEpfkTIE0jgwvUKNA&oe=69A7DAD0&_nc_sid=201bca (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1017)')))


The cache directory has been deleted
Configuration ready. Replace 'YOUR_MAPILLARY_API_KEY_HERE' and uncomment to download images.


## 4. Processing Panoramic Images

Convert panoramic street view images to perspective views and analyze them.


In [6]:
# Configure for panoramic image processing
config_panorama = PipelineConfig(
    input_dir="sf/mly_svi/batch_1",
    output_dir="sf/mly_svi/output_pano",
    processing_config=ProcessingConfig(
        is_panorama=True,           # Enable panorama preprocessing
        panorama_fov=90,           # Field of view (degrees)
        panorama_phi=0.0,           # Vertical angle
        panorama_aspects=(10, 10),  # Aspect ratio
        panorama_show_size=100,     # Scale factor
        log_level="INFO"
    ),
    viz_config=VisualizationConfig(
        save_visualizations=True,
        save_corrected_images=True,
        save_intermediate_results=True
    )
)

print("✓ Panorama pipeline configured")
print(f"  Input:  {config_panorama.input_dir}")
print(f"  Output: {config_panorama.output_dir}")


✓ Panorama pipeline configured
  Input:  /Users/cubics/Vision2Slope/examples/sf/mly_svi/batch_1
  Output: /Users/cubics/Vision2Slope/examples/sf/mly_svi/output_pano


In [7]:
# Process panoramic images
pipeline_panorama = Vision2SlopePipeline(config_panorama)
results_panorama = pipeline_panorama.process_batch()

print(f"\n✓ Panorama processing completed!")
print(f"  Total images: {len(results_panorama)}")
print(f"  Successful:   {len(results_panorama[results_panorama['status'] == 'success'])}")

if len(results_panorama) > 0:
    print(f"\nResults saved to: {config_panorama.output_dir}")

INFO - ============================================================
INFO - PANORAMA PREPROCESSING
INFO - ============================================================
INFO - Input directory (panoramic images): /Users/cubics/Vision2Slope/examples/sf/mly_svi/batch_1
INFO - Output directory (perspective views): /Users/cubics/Vision2Slope/examples/sf/mly_svi/output_pano/panorama_perspectives
INFO - Transforming panoramic images from /Users/cubics/Vision2Slope/examples/sf/mly_svi/batch_1
INFO - Output directory: /Users/cubics/Vision2Slope/examples/sf/mly_svi/output_pano/panorama_perspectives
WARNING - generate_left_right=False: transforming all images as-is
/Users/cubics/Vision2Slope/examples/sf/mly_svi/output_pano/panorama_perspectives


Converting to perspective: 100%|██████████| 11/11 [00:01<00:00, 10.53it/s]


✓ Cleaned up 22 redundant perspective images
INFO - Generated 0 perspective views
INFO - Panorama preprocessing completed successfully
INFO - ============================================================
INFO - Updated input directory to: /Users/cubics/Vision2Slope/examples/sf/mly_svi/output_pano/panorama_perspectives/perspective
INFO - Initializing pipeline components...
INFO - Loading segmentation model: facebook/mask2former-swin-large-mapillary-vistas-semantic


INFO - Model loaded successfully on device: cpu
INFO - Pipeline components initialized successfully
INFO - Found 22 images to process


Processing images:   0%|          | 0/22 [00:00<?, ?it/s]

...

INFO - Successfully processed 802263434204153_Direction_90_FOV_90_aspect_10--10_raw.png


Processing images: 100%|██████████| 22/22 [01:21<00:00,  3.72s/it]

INFO - Results saved to: /Users/cubics/Vision2Slope/examples/sf/mly_svi/output_pano/vision2slope_results_20260202_151543.csv
INFO - Found 12 valid segments for bi-directional estimation



Computing road slopes: 100%|██████████| 10/10 [00:00<00:00, 9295.89it/s]

INFO - Estimated results saved to: /Users/cubics/Vision2Slope/examples/sf/mly_svi/output_pano/vision2slope_results_20260202_151543_estimate.csv
INFO - Intermediate results saved to: /Users/cubics/Vision2Slope/examples/sf/mly_svi/output_pano/intermediate_results
INFO - ============================================================
INFO - VISION2SLOPE PROCESSING SUMMARY
INFO - ============================================================
INFO - Total images: 12
INFO -   success: 12 (100.0%)
INFO - 
Successful Results Statistics:
INFO -   Avg skew angle: -3.54°
INFO -   Avg road edge line slope: 0.0722
INFO -   Avg road edge line angle: 4.10°
INFO -   Avg road area: 221778 pixels
INFO -   Avg road estimated slope: 8.61°
INFO - ============================================================

✓ Panorama processing completed!
  Total images: 12
  Successful:   12

Results saved to: /Users/cubics/Vision2Slope/examples/sf/mly_svi/output_pano


## 4. Basic Usage - Processing Perspective Images

Process regular street view images (non-panoramic format).


In [5]:
# Configure for processing perspective images
config_perspective = PipelineConfig(
    input_dir="input",
    output_dir="output"
)

print("✓ Configuration created for perspective images")
print(f"  Input:  {config_perspective.input_dir}")
print(f"  Output: {config_perspective.output_dir}")


✓ Configuration created for perspective images
  Input:  /Users/cubics/Vision2Slope/examples/input
  Output: /Users/cubics/Vision2Slope/examples/output


In [ ]:
# Run pipeline on perspective images
# Ensure you have image files in the 'input' directory before running this cell

pipeline_perspective = Vision2SlopePipeline(config_perspective)
results_perspective = pipeline_perspective.process_batch()

print(f"\n✓ Processing completed!")
print(f"  Total images: {len(results_perspective)}")
print(f"  Successful:   {len(results_perspective[results_perspective['status'] == 'success'])}")

if len(results_perspective) > 0:
    print(f"\nResults saved to: {config_perspective.output_dir}")


## 6. Analyzing Results

Examine and visualize the processing results.


In [ ]:
# Display summary statistics from results
if len(results_panorama) > 0:
    successful_df = results_panorama[results_panorama['status'] == 'success']
    
    if len(successful_df) > 0:
        print("📊 Road Slope Analysis Results\n")
        print(f"Total successful images: {len(successful_df)}")
        print(f"Average skew angle: {successful_df['skew_angle'].mean():.2f}°")
        print(f"Average road edge angle: {successful_df['road_edge_line_angle'].mean():.2f}°")
        print(f"Average road area: {successful_df['road_area'].mean():.0f} pixels")
        
        if 'road_estimated_slope' in successful_df.columns:
            print(f"Average estimated slope: {successful_df['road_estimated_slope'].mean():.2f}°")
        
        # Display first few results
        print("\n📋 First few results:")
        display(successful_df[['filename', 'skew_angle', 'road_edge_line_angle', 'status']].head())
